# Merge likely over-split units in one SpikeInterface session

This workflow is independent of cross-session stitching. It runs UnitMatch on the two temporal halves of one recording, requires both directional match probabilities to exceed the threshold, rejects pairs that fail the ISI safety check, and applies disjoint merges with SpikeInterface's `soft` mode.

Soft merging does not require raw traces. Run this workflow again on the merged analyzer if you want to discover additional over-splits.

In [ ]:
from pathlib import Path

USE_SYNTHETIC_DATA = True
ANALYZER_PATH = Path("path/to/sorting_analyzer")
EXPORT_DIR = Path("unitmatch_single_session")
MATCH_THRESHOLD = 0.5
CENSORED_PERIOD_MS = 0.5
APPLY_MERGES = True

In [ ]:
import json

import numpy as np
import spikeinterface.full as si

import UnitMatchPy.bayes_functions as bf
import UnitMatchPy.overlord as ov
import UnitMatchPy.utils as util
from UnitMatchPy.assign_unique_id import get_within_session_merge_groups
from UnitMatchPy.default_params import get_default_param
from UnitMatchPy.save_utils import make_UnitMatch_folder_from_sorting_analyzers


def make_synthetic_analyzer(seed=410):
    recording, sorting = si.generate_ground_truth_recording(
        durations=[5.0], sampling_frequency=30_000.0,
        num_channels=16, num_units=8, seed=seed,
    )
    analyzer = si.create_sorting_analyzer(
        sorting=sorting, recording=recording, format="memory", sparse=True
    )
    analyzer.compute(
        "random_spikes", method="uniform", max_spikes_per_unit=200, seed=seed
    )
    analyzer.compute("waveforms", ms_before=1.0, ms_after=1.0)
    return analyzer


analyzer = (
    make_synthetic_analyzer()
    if USE_SYNTHETIC_DATA
    else si.load_sorting_analyzer(ANALYZER_PATH)
)
required_extensions = {"random_spikes", "waveforms"}
missing_extensions = [
    name for name in required_extensions if not analyzer.has_extension(name)
]
if missing_extensions:
    raise ValueError(f"Analyzer is missing extensions: {missing_extensions}")

In [ ]:
make_UnitMatch_folder_from_sorting_analyzers(
    [analyzer], EXPORT_DIR, overwrite=True
)
wave_path = EXPORT_DIR / "Session0"
channel_pos = [np.load(wave_path / "channel_locations.npy")]
with (wave_path / "waveform_params.json").open(encoding="utf-8") as stream:
    waveform_params = json.load(stream)

param = get_default_param()
param.update(waveform_params)
param["waveidx"] = np.asarray(param["waveidx"], dtype=int)
param["match_threshold"] = MATCH_THRESHOLD
param = util.get_probe_geometry(channel_pos[0], param)

unit_ids_per_session = [analyzer.unit_ids]
waveform, session_id, session_switch, within_session, unit_ids, param = (
    util.load_waveforms([wave_path], unit_ids_per_session, param)
)
clus_info = {
    "good_units": unit_ids,
    "session_switch": session_switch,
    "session_id": session_id,
    "original_ids": np.concatenate(unit_ids),
    "spike_times": [
        analyzer.sorting.get_unit_spike_train(unit_id=unit_id)
        / analyzer.sampling_frequency
        for unit_id in analyzer.unit_ids
    ],
}

properties = ov.extract_parameters(waveform, channel_pos, clus_info, param)
total_score, candidate_pairs, scores, predictors = ov.extract_metric_scores(
    properties, session_switch, within_session, param, niter=2
)
prior_match = 1 - param["n_expected_matches"] / param["n_units"] ** 2
priors = np.array((prior_match, 1 - prior_match))
labels = candidate_pairs.astype(int)
conditions = np.unique(labels)
kernels = bf.get_parameter_kernels(scores, labels, conditions, param, add_one=1)
probability = bf.apply_naive_bayes(
    kernels, priors, predictors, param, conditions
)
output_prob_matrix = probability[:, 1].reshape(param["n_units"], param["n_units"])

In [ ]:
merge_groups = get_within_session_merge_groups(
    output_prob_matrix, param, clus_info, match_threshold=MATCH_THRESHOLD
)
print(f"Proposed merge groups: {merge_groups}")

if APPLY_MERGES and merge_groups:
    merged_analyzer = analyzer.merge_units(
        merge_unit_groups=merge_groups,
        censored_period_ms=CENSORED_PERIOD_MS,
        merging_mode="soft",
    )
else:
    merged_analyzer = analyzer

print(f"Units before: {len(analyzer.unit_ids)}")
print(f"Units after:  {len(merged_analyzer.unit_ids)}")